# Basic data cleaning (Information Systems)

Rule-based: clean author names, fill missing values, similarity-based standardization of addresses. No API.

In [1]:
import pandas as pd
import re
from pathlib import Path

# Load Information Systems article data (same folder as this notebook)
CSV_PATH = Path("Information_Systems_article_data.csv")
df = pd.read_csv(CSV_PATH)

# Convert to lowercase and remove special characters for authors
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df["Standardized_name"] = df["Author_name"].apply(clean_text)
print(df.head())

                                                 URL        Journal_Title  \
0  https://www.sciencedirect.com/science/article/...  Information Systems   
1  https://www.sciencedirect.com/science/article/...  Information Systems   
2  https://www.sciencedirect.com/science/article/...  Information Systems   
3  https://www.sciencedirect.com/science/article/...  Information Systems   
4  https://www.sciencedirect.com/science/article/...  Information Systems   

                                       Article_Title Volume_Issue Month_Year  \
0  Efficient data structures for fast and low-cos...   Volume 139  July 2026   
1  Efficient data structures for fast and low-cos...   Volume 139  July 2026   
2  Efficient data structures for fast and low-cos...   Volume 139  July 2026   
3  Measuring the decentralisation of DeFi develop...   Volume 139  July 2026   
4  Measuring the decentralisation of DeFi develop...   Volume 139  July 2026   

   Abstract                                           Ke

In [2]:
df = df.fillna('NULL')

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert addresses to TF-IDF vectors (character n-grams) to find similar ones
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
tfidf_matrix = vectorizer.fit_transform(df["Author_Address"].fillna(''))
print(tfidf_matrix.shape)

(4718, 30529)


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_df = pd.DataFrame(similarity_matrix, index=df["Author_Address"].fillna(''), columns=df["Author_Address"].fillna(''))
print(similarity_df.iloc[:5, :5])

Author_Address                                      University of New South Wales, Sydney, 2032, USW, Australia  \
Author_Address                                                                                                    
University of New South Wales, Sydney, 2032, US...                                           1.000000             
University of New South Wales, Sydney, 2032, US...                                           1.000000             
Newcastle University, Tyne and Wear, England, U...                                           0.050592             
Department of Computer Science, University Coll...                                           0.033336             
Department of Computer Science, University Coll...                                           0.033336             

Author_Address                                      University of New South Wales, Sydney, 2032, USW, Australia  \
Author_Address                                                                 

In [5]:
threshold = 0.75
matches = {}
addr = df["Author_Address"].fillna('')
for i in range(len(addr)):
    for j in range(len(addr)):
        if i != j and similarity_matrix[i][j] > threshold:
            key = addr.iloc[i]
            matches.setdefault(key, []).append(addr.iloc[j])
matches_df = pd.DataFrame(list(matches.items()), columns=["Original Address", "Similar Address"])
print(matches_df.head(10))

                                    Original Address  \
0  University of New South Wales, Sydney, 2032, U...   
1  Department of Computer Science, University Col...   
2  Harbin Engineering University, 145 Nantong Str...   
3     Arizona State University, Tempe, United States   
4           Shahid Beheshti University, Tehran, Iran   
5                         KU Leuven, Leuven, Belgium   
6  Free University of Bozen-Bolzano, Faculty of E...   
7  Technical University of Munich, School of Comp...   
8  RWTH Aachen, Chair of Process and Data Science...   
9       Kühne Logistics University, Hamburg, Germany   

                                     Similar Address  
0  [University of New South Wales, Sydney, 2032, ...  
1  [Department of Computer Science, University Co...  
2  [Harbin Engineering University, 145 Nantong St...  
3          [Arizona State University, United States]  
4  [Department of Information Technology Manageme...  
5  [KU Leuven, Belgium, KU Leuven, Belgium, KU Le... 

In [6]:
def standardize_name(name, matches):
    if name in matches and isinstance(matches[name], list):
        valid = [str(n) for n in matches[name] if pd.notna(n)]
        if valid:
            return min(valid, key=len)
    return name

df["Standardized_Address"] = df["Author_Address"].fillna('').apply(lambda x: standardize_name(x, matches))
df.to_csv("Information_Systems_basic_cleaned.csv", index=False)
print(df[["Author_Address", "Standardized_Address"]].head())

                                      Author_Address  \
0  University of New South Wales, Sydney, 2032, U...   
1  University of New South Wales, Sydney, 2032, U...   
2  Newcastle University, Tyne and Wear, England, ...   
3  Department of Computer Science, University Col...   
4  Department of Computer Science, University Col...   

                                Standardized_Address  
0   University of New South Wales, Sydney, Australia  
1   University of New South Wales, Sydney, Australia  
2  Newcastle University, Tyne and Wear, England, ...  
3              University College London, London, UK  
4              University College London, London, UK  
